In [ ]:
!pip install kagglehub
!pip install xgboost lightgbm catboost imbalanced-learn shap


In [ ]:
import pandas as pd
import os

df = pd.read_csv("/content/creditcard.csv")
df.head()

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0.0
1,0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0.0
2,1,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0.0
3,1,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0.0
4,2,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0.0


In [ ]:
df = df.rename(columns={'Class':'label'})

df = df.dropna(subset=['label'])

X = df.drop("label", axis=1)
y = df["label"]

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Train–test split with class stratification
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
from imblearn.over_sampling import SMOTE

sm = SMOTE(random_state=42, k_neighbors=1)
X_train_res, y_train_res = sm.fit_resample(X_train_scaled, y_train)

In [ ]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix
)

def evaluate_model(model, X_test, y_test, model_name="Model"):
    pred = model.predict(X_test)
    if hasattr(model, "predict_proba"):
        prob = model.predict_proba(X_test)[:,1]
    elif hasattr(model, "decision_function"):
        prob = model.decision_function(X_test)
    else:
        prob = pred

    print(f"\n===== {model_name} =====")
    print("Accuracy:", accuracy_score(y_test, pred))
    print("Precision:", precision_score(y_test, pred))
    print("Recall:", recall_score(y_test, pred))
    print("F1 Score:", f1_score(y_test, pred))
    print("ROC-AUC:", roc_auc_score(y_test, prob))
    print("PR-AUC:", average_precision_score(y_test, prob))
    print("Confusion Matrix:\n", confusion_matrix(y_test, pred))

    return {
        "Model": model_name,
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred),
        "Recall": recall_score(y_test, pred),
        "F1": f1_score(y_test, pred),
        "ROC-AUC": roc_auc_score(y_test, prob),
        "PR-AUC": average_precision_score(y_test, prob)
    }


In [ ]:
results = []


In [ ]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(class_weight="balanced", max_iter=2000)
lr.fit(X_train_res, y_train_res)

results.append(evaluate_model(lr, X_test_scaled, y_test, "Logistic Regression"))



===== Logistic Regression =====
Accuracy: 0.9993714644877436
Precision: 0.5
Recall: 1.0
F1 Score: 0.6666666666666666
ROC-AUC: 1.0
PR-AUC: 1.0
Confusion Matrix:
 [[1589    1]
 [   0    1]]


In [ ]:
from sklearn.tree import DecisionTreeClassifier

dt = DecisionTreeClassifier(class_weight="balanced", random_state=42)
dt.fit(X_train, y_train)

results.append(evaluate_model(dt, X_test, y_test, "Decision Tree"))


In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=200,
    class_weight="balanced_subsample",
    random_state=42
)
rf.fit(X_train, y_train)

results.append(evaluate_model(rf, X_test, y_test, "Random Forest"))


In [ ]:
from xgboost import XGBClassifier

pos_weight = (len(y_train)-sum(y_train)) / sum(y_train)

xgb = XGBClassifier(
    eval_metric='logloss',
    scale_pos_weight=pos_weight,
    random_state=42
)
xgb.fit(X_train, y_train)

results.append(evaluate_model(xgb, X_test, y_test, "XGBoost"))


In [ ]:
import lightgbm as lgb

lgbm = lgb.LGBMClassifier(
    is_unbalance=True,
    random_state=42
)
lgbm.fit(X_train, y_train)

results.append(evaluate_model(lgbm, X_test, y_test, "LightGBM"))


In [ ]:
from catboost import CatBoostClassifier

cb = CatBoostClassifier(
    iterations=300,
    depth=6,
    learning_rate=0.1,
    loss_function="Logloss",
    random_seed=42,
    verbose=False,
    class_weights=[1, sum(y_train==0)/sum(y_train==1)]
)
cb.fit(X_train, y_train)

results.append(evaluate_model(cb, X_test, y_test, "CatBoost"))


In [ ]:
from sklearn.naive_bayes import GaussianNB

nb = GaussianNB()
nb.fit(X_train_res, y_train_res)

results.append(evaluate_model(nb, X_test_scaled, y_test, "Naive Bayes"))


In [ ]:
from sklearn.neural_network import MLPClassifier

mlp = MLPClassifier(hidden_layer_sizes=(64,32), max_iter=500, random_state=42)
mlp.fit(X_train_res, y_train_res)

results.append(evaluate_model(mlp, X_test_scaled, y_test, "MLP Neural Network"))


In [ ]:
from sklearn.ensemble import IsolationForest

iso = IsolationForest(contamination=sum(y)/len(y), random_state=42)
iso.fit(X_train_scaled[y_train==0])

pred = iso.predict(X_test_scaled)
pred = [1 if p==-1 else 0 for p in pred]

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
results.append({
    "Model": "Isolation Forest",
    "Accuracy": accuracy_score(y_test, pred),
    "Precision": precision_score(y_test, pred),
    "Recall": recall_score(y_test, pred),
    "F1": f1_score(y_test, pred),
    "ROC-AUC": roc_auc_score(y_test, pred),
    "PR-AUC": average_precision_score(y_test, pred)
})


In [ ]:
from sklearn.svm import OneClassSVM

ocsvm = OneClassSVM(kernel="rbf", nu=0.01, gamma="scale")
ocsvm.fit(X_train_scaled[y_train==0])

pred = ocsvm.predict(X_test_scaled)
pred = [1 if p==-1 else 0 for p in pred]

results.append({
    "Model": "One-Class SVM",
    "Accuracy": accuracy_score(y_test, pred),
    "Precision": precision_score(y_test, pred),
    "Recall": recall_score(y_test, pred),
    "F1": f1_score(y_test, pred),
    "ROC-AUC": roc_auc_score(y_test, pred),
    "PR-AUC": average_precision_score(y_test, pred)
})


In [ ]:
import pandas as pd

results_df = pd.DataFrame(results)
results_df


In [ ]:
import matplotlib.pyplot as plt

results_df.set_index("Model")[["ROC-AUC","PR-AUC","F1"]].plot(kind="bar", figsize=(14,6))
plt.title("Model Comparison on Fraud Detection")
plt.show()


In [ ]:
from sklearn.metrics import roc_curve, precision_recall_curve

models = {
    "Logistic Regression": lr,
    "Decision Tree": dt,
    "Random Forest": rf,
    "XGBoost": xgb,
    "LightGBM": lgbm,
    "CatBoost": cb,
    "Naive Bayes": nb,
    "MLP Neural Network": mlp
}

In [ ]:
plt.figure(figsize=(10,6))
for name, model in models.items():
    if name in ["Isolation Forest", "One-Class SVM"]: continue
    y_prob = model.predict_proba(X_test_scaled)[:,1]
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    plt.plot(fpr, tpr, label=name)
plt.plot([0,1],[0,1],'k--')
plt.xlabel("FPR"); plt.ylabel("TPR"); plt.title("ROC Curves")
plt.legend(); plt.savefig("roc_all_models.png"); plt.show()

plt.figure(figsize=(10,6))
for name, model in models.items():
    if name in ["Isolation Forest", "One-Class SVM"]: continue
    y_prob = model.predict_proba(X_test_scaled)[:,1]
    precision, recall, _ = precision_recall_curve(y_test, y_prob)
    plt.plot(recall, precision, label=name)
plt.xlabel("Recall"); plt.ylabel("Precision"); plt.title("PR Curves")
plt.legend(); plt.savefig("pr_all_models.png"); plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(10,6))
sns.heatmap(results_df.set_index("Model").astype(float), annot=True, cmap="YlGnBu")
plt.title("Model Comparison Heatmap")
plt.savefig("heatmap_comparison.png")
plt.show()

In [ ]:
# Extra Trees

!pip install pytorch-tabnet
!pip install scikit-learn imbalanced-learn

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.ensemble import ExtraTreesClassifier
from imblearn.over_sampling import SMOTE

from pytorch_tabnet.tab_model import TabNetClassifier
import torch

df = pd.read_csv("/content/creditcard.csv")
X = df.drop("Class", axis=1)
y = df["Class"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Balance classes with SMOTE for supervised learning
sm = SMOTE(random_state=42)
X_train_res, y_train_res = sm.fit_resample(X_train_scaled, y_train)

# Extra Trees Classifier
et_model = ExtraTreesClassifier(n_estimators=200, max_depth=None, random_state=42, class_weight="balanced")
et_model.fit(X_train_res, y_train_res)
y_pred_et = et_model.predict(X_test_scaled)

# Metrics
acc_et = accuracy_score(y_test, y_pred_et)
prec_et = precision_score(y_test, y_pred_et)
rec_et = recall_score(y_test, y_pred_et)
f1_et = f1_score(y_test, y_pred_et)
roc_et = roc_auc_score(y_test, y_pred_et)

print("Extra Trees Results:")
print(f"Accuracy: {acc_et:.4f}, Precision: {prec_et:.4f}, Recall: {rec_et:.4f}, F1: {f1_et:.4f}, ROC-AUC: {roc_et:.4f}")

# TabNet Classifier
tabnet_model = TabNetClassifier(seed=42, verbose=0)
tabnet_model.fit(
    X_train_res, y_train_res,
    eval_set=[(X_test_scaled, y_test)],
    max_epochs=50,
    patience=10,
    batch_size=1024,
    virtual_batch_size=128
)
y_pred_tabnet = tabnet_model.predict(X_test_scaled)

# Metrics
acc_tab = accuracy_score(y_test, y_pred_tabnet)
prec_tab = precision_score(y_test, y_pred_tabnet)
rec_tab = recall_score(y_test, y_pred_tabnet)
f1_tab = f1_score(y_test, y_pred_tabnet)
roc_tab = roc_auc_score(y_test, y_pred_tabnet)

print("TabNet Results:")
print(f"Accuracy: {acc_tab:.4f}, Precision: {prec_tab:.4f}, Recall: {rec_tab:.4f}, F1: {f1_tab:.4f}, ROC-AUC: {roc_tab:.4f}")


In [ ]:
# Compute PR-AUC for both
from sklearn.metrics import precision_recall_curve, auc
import pandas as pd

# Extra Trees PR-AUC
y_prob_et = et_model.predict_proba(X_test_scaled)[:,1]
precision_et, recall_et, _ = precision_recall_curve(y_test, y_prob_et)
pr_auc_et = auc(recall_et, precision_et)

# TabNet PR-AUC
y_prob_tabnet = tabnet_model.predict_proba(X_test_scaled)[:,1]
precision_tab, recall_tab, _ = precision_recall_curve(y_test, y_prob_tabnet)
pr_auc_tab = auc(recall_tab, precision_tab)

new_rows = pd.DataFrame([
    {
        "Model": "Extra Trees",
        "Accuracy": acc_et,
        "Precision": prec_et,
        "Recall": rec_et,
        "F1": f1_et,
        "ROC-AUC": roc_et,
        "PR-AUC": pr_auc_et
    },
    {
        "Model": "TabNet",
        "Accuracy": acc_tab,
        "Precision": prec_tab,
        "Recall": rec_tab,
        "F1": f1_tab,
        "ROC-AUC": roc_tab,
        "PR-AUC": pr_auc_tab
    }
])

results_df = pd.concat([results_df, new_rows], ignore_index=True)

# 2. Plot Comparison
import matplotlib.pyplot as plt

results_df.set_index("Model")[["ROC-AUC","PR-AUC","F1"]].plot(kind="bar", figsize=(14,6))
plt.title("Model Comparison on Fraud Detection")
plt.ylabel("Score")
plt.xticks(rotation=45)
plt.grid(axis='y')
plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Heatmap of all metrics including Extra Trees and TabNet
plt.figure(figsize=(12,8))

heatmap_data = results_df.set_index("Model").astype(float)

sns.heatmap(heatmap_data, annot=True, fmt=".4f", cmap="YlGnBu", cbar=True)
plt.title("Model Comparison Heatmap")
plt.ylabel("Model")
plt.xlabel("Metric")
plt.xticks(rotation=45)
plt.tight_layout()

# Save figure
plt.savefig("heatmap_comparison.png")
plt.show()


In [ ]:
# K-Nearest Neighbors (KNN)
from sklearn.neighbors import KNeighborsClassifier

from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Balance classes with SMOTE
from imblearn.over_sampling import SMOTE
sm = SMOTE(random_state=42)
X_train_res, y_train_res = sm.fit_resample(X_train_scaled, y_train)

# Initialize KNN
knn_model = KNeighborsClassifier(n_neighbors=5, metric='minkowski', weights='uniform')
knn_model.fit(X_train_res, y_train_res)

# Predictions
y_pred_knn = knn_model.predict(X_test_scaled)
y_prob_knn = knn_model.predict_proba(X_test_scaled)[:,1]

# Metrics
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, precision_recall_curve, auc

precision_knn, recall_knn, _ = precision_recall_curve(y_test, y_prob_knn)
pr_auc_knn = auc(recall_knn, precision_knn)

results_knn = {
    "Model": "KNN",
    "Accuracy": accuracy_score(y_test, y_pred_knn),
    "Precision": precision_score(y_test, y_pred_knn),
    "Recall": recall_score(y_test, y_pred_knn),
    "F1": f1_score(y_test, y_pred_knn),
    "ROC-AUC": roc_auc_score(y_test, y_pred_knn),
    "PR-AUC": pr_auc_knn
}

if 'results_df' in globals():
    results_df = pd.concat([results_df, pd.DataFrame([results_knn])], ignore_index=True)
else:
    results_df = pd.DataFrame([results_knn])

print("KNN Results:")
print(results_knn)

plt.figure(figsize=(12,6))
sns.heatmap(results_df.set_index("Model").astype(float), annot=True, fmt=".4f", cmap="YlGnBu")
plt.title("Model Comparison Heatmap (Including KNN)")
plt.show()

In [ ]:
pip install scikit-learn xgboost lightgbm catboost imbalanced-learn seaborn matplotlib


In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import OneClassSVM
from sklearn.metrics import roc_auc_score
import xgboost as xgb
import lightgbm as lgb
import catboost as cb

df = pd.read_csv("/content/creditcard.csv")
X = df.drop("Class", axis=1).values
y = df["Class"].values

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Balance the dataset using SMOTE
sm = SMOTE(random_state=42)
X_train_res, y_train_res = sm.fit_resample(X_train_scaled, y_train)

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=42),
    "XGBoost": xgb.XGBClassifier(random_state=42),
    "LightGBM": lgb.LGBMClassifier(random_state=42),
    "CatBoost": cb.CatBoostClassifier(learning_rate=0.1, iterations=200, depth=7, verbose=0),
    "Naive Bayes": GaussianNB(),
    "MLP Neural Network": MLPClassifier(hidden_layer_sizes=(64,32), max_iter=500, random_state=42),
    "Isolation Forest": IsolationForest(random_state=42),
    "One Class SVM": OneClassSVM(nu=0.1, kernel="rbf", gamma="scale"),
    "Extra Trees": ExtraTreesClassifier(n_estimators=200, class_weight="balanced", random_state=42),
    "KNN": KNeighborsClassifier(n_neighbors=5)
}


def plot_confusion_matrix(model_name, model, X_train, y_train, X_test, y_test):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    # Confusion Matrix
    cm = confusion_matrix(y_test, y_pred)

    plt.figure(figsize=(6,6))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["Normal", "Fraud"], yticklabels=["Normal", "Fraud"])
    plt.title(f"Confusion Matrix: {model_name}")
    plt.xlabel("Predicted Label")
    plt.ylabel("True Label")
    plt.show()

    # Classification Report
    print(f"Classification Report for {model_name}:")
    print(classification_report(y_test, y_pred))

for model_name, model in models.items():
    plot_confusion_matrix(model_name, model, X_train_res, y_train_res, X_test_scaled, y_test)



In [ ]:
!pip install -q xgboost lightgbm catboost imbalanced-learn pytorch-tabnet shap

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix,
    precision_recall_curve, roc_curve, auc
)

BASE = "/content/results"
os.makedirs(BASE, exist_ok=True)
os.makedirs(os.path.join(BASE, "plots"), exist_ok=True)
os.makedirs(os.path.join(BASE, "confusion_matrices"), exist_ok=True)
os.makedirs(os.path.join(BASE, "feature_importance"), exist_ok=True)
os.makedirs(os.path.join(BASE, "models"), exist_ok=True)


In [ ]:
df = pd.read_csv("/content/creditcard.csv")
df = df.rename(columns={'Class':'label'})
df = df.dropna(subset=['label'])

X = df.drop("label", axis=1)
y = df["label"]

# Train–test split with class stratification
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# SMOTE for training
sm = SMOTE(random_state=42)
X_train_res, y_train_res = sm.fit_resample(X_train_scaled, y_train)

df_stats = {
    "total_rows": len(df),
    "total_frauds": int(df['label'].sum()),
    "total_nonfrauds": int(len(df)-df['label'].sum())
}
pd.DataFrame([df_stats]).to_csv(os.path.join(BASE, "dataset_stats.csv"), index=False)

plt.figure(figsize=(6,4))
ax = sns.countplot(x=df['label'])
plt.title("Dataset Fraud Distribution (label)")
plt.xlabel("label")
plt.ylabel("count")


for p in ax.patches:
    height = p.get_height()
    ax.text(p.get_x() + p.get_width()/2., height + 50, int(height), ha="center")

plt.savefig(os.path.join(BASE, "plots", "dataset_fraud_distribution.png"), bbox_inches='tight')
plt.show()



In [ ]:
def evaluate_model(model, X_test, y_test, model_name="Model"):
    pred = model.predict(X_test)
    if hasattr(model, "predict_proba"):
        prob = model.predict_proba(X_test)[:,1]
    elif hasattr(model, "decision_function"):
        prob = model.decision_function(X_test)
    else:
        prob = pred

    print(f"\n===== {model_name} =====")
    print("Accuracy:", accuracy_score(y_test, pred))
    print("Precision:", precision_score(y_test, pred, zero_division=0))
    print("Recall:", recall_score(y_test, pred, zero_division=0))
    print("F1 Score:", f1_score(y_test, pred, zero_division=0))
    try:
        roc = roc_auc_score(y_test, prob)
    except Exception as e:
        roc = np.nan
    try:
        pr = average_precision_score(y_test, prob)
    except Exception as e:
        pr = np.nan

    print("ROC-AUC:", roc)
    print("PR-AUC:", pr)
    print("Confusion Matrix:\n", confusion_matrix(y_test, pred))

    return {
        "Model": model_name,
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred, zero_division=0),
        "Recall": recall_score(y_test, pred, zero_division=0),
        "F1": f1_score(y_test, pred, zero_division=0),
        "ROC-AUC": roc,
        "PR-AUC": pr
    }

results = []


In [ ]:
# 1) Logistic Regression
from sklearn.linear_model import LogisticRegression
lr = LogisticRegression(class_weight="balanced", max_iter=2000)
lr.fit(X_train_res, y_train_res)
results.append(evaluate_model(lr, X_test_scaled, y_test, "Logistic Regression"))

# 2) Decision Tree
from sklearn.tree import DecisionTreeClassifier
dt = DecisionTreeClassifier(class_weight="balanced", random_state=42)
dt.fit(X_train, y_train)
results.append(evaluate_model(dt, X_test, y_test, "Decision Tree"))

# 3) Random Forest
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier(
    n_estimators=200,
    class_weight="balanced_subsample",
    random_state=42
)
rf.fit(X_train, y_train)
results.append(evaluate_model(rf, X_test, y_test, "Random Forest"))

# 4) XGBoost
from xgboost import XGBClassifier
pos_weight = (len(y_train)-sum(y_train)) / sum(y_train)
xgb = XGBClassifier(
    eval_metric='logloss',
    scale_pos_weight=pos_weight,
    random_state=42,
    use_label_encoder=False
)
xgb.fit(X_train, y_train)
results.append(evaluate_model(xgb, X_test, y_test, "XGBoost"))

# 5) LightGBM
import lightgbm as lgb
lgbm = lgb.LGBMClassifier(
    is_unbalance=True,
    random_state=42
)
lgbm.fit(X_train, y_train)
results.append(evaluate_model(lgbm, X_test, y_test, "LightGBM"))

# 6) Naive Bayes
from sklearn.naive_bayes import GaussianNB
nb = GaussianNB()
nb.fit(X_train_res, y_train_res)
results.append(evaluate_model(nb, X_test_scaled, y_test, "Naive Bayes"))

# 7) MLP Neural Network
from sklearn.neural_network import MLPClassifier
mlp = MLPClassifier(hidden_layer_sizes=(64,32), max_iter=500, random_state=42)
mlp.fit(X_train_res, y_train_res)
results.append(evaluate_model(mlp, X_test_scaled, y_test, "MLP Neural Network"))

# 8) Isolation Forest
from sklearn.ensemble import IsolationForest
iso = IsolationForest(contamination=sum(y)/len(y), random_state=42)
iso.fit(X_train_scaled[y_train==0])
pred_iso = iso.predict(X_test_scaled)
pred_iso = np.array([1 if p==-1 else 0 for p in pred_iso])

results.append({
    "Model": "Isolation Forest",
    "Accuracy": accuracy_score(y_test, pred_iso),
    "Precision": precision_score(y_test, pred_iso, zero_division=0),
    "Recall": recall_score(y_test, pred_iso, zero_division=0),
    "F1": f1_score(y_test, pred_iso, zero_division=0),
    "ROC-AUC": np.nan,
    "PR-AUC": np.nan
})

# 9) One-Class SVM
from sklearn.svm import OneClassSVM
ocsvm = OneClassSVM(kernel="rbf", nu=0.01, gamma="scale")
ocsvm.fit(X_train_scaled[y_train==0])
pred_ocsvm = ocsvm.predict(X_test_scaled)
pred_ocsvm = np.array([1 if p==-1 else 0 for p in pred_ocsvm])
results.append({
    "Model": "One-Class SVM",
    "Accuracy": accuracy_score(y_test, pred_ocsvm),
    "Precision": precision_score(y_test, pred_ocsvm, zero_division=0),
    "Recall": recall_score(y_test, pred_ocsvm, zero_division=0),
    "F1": f1_score(y_test, pred_ocsvm, zero_division=0),
    "ROC-AUC": np.nan,
    "PR-AUC": np.nan
})

# 10) Extra Trees
from sklearn.ensemble import ExtraTreesClassifier
et_model = ExtraTreesClassifier(n_estimators=200, max_depth=None, random_state=42, class_weight="balanced")
et_model.fit(X_train_res, y_train_res)
y_pred_et = et_model.predict(X_test_scaled)
results.append({
    "Model": "Extra Trees",
    "Accuracy": accuracy_score(y_test, y_pred_et),
    "Precision": precision_score(y_test, y_pred_et, zero_division=0),
    "Recall": recall_score(y_test, y_pred_et, zero_division=0),
    "F1": f1_score(y_test, y_pred_et, zero_division=0),
    "ROC-AUC": np.nan,
    "PR-AUC": np.nan
})

# 11) TabNet
from pytorch_tabnet.tab_model import TabNetClassifier
tabnet_model = TabNetClassifier(seed=42, verbose=0)
tabnet_model.fit(
    X_train_res, y_train_res,
    eval_set=[(X_test_scaled, y_test)],
    max_epochs=50,
    patience=10,
    batch_size=1024,
    virtual_batch_size=128
)
y_pred_tabnet = tabnet_model.predict(X_test_scaled)
results.append({
    "Model": "TabNet",
    "Accuracy": accuracy_score(y_test, y_pred_tabnet),
    "Precision": precision_score(y_test, y_pred_tabnet, zero_division=0),
    "Recall": recall_score(y_test, y_pred_tabnet, zero_division=0),
    "F1": f1_score(y_test, y_pred_tabnet, zero_division=0),
    "ROC-AUC": np.nan,
    "PR-AUC": np.nan
})

# 12) KNN
from sklearn.neighbors import KNeighborsClassifier
knn_model = KNeighborsClassifier(n_neighbors=5, metric='minkowski', weights='uniform')
knn_model.fit(X_train_res, y_train_res)
y_pred_knn = knn_model.predict(X_test_scaled)
y_prob_knn = knn_model.predict_proba(X_test_scaled)[:,1]
precision_knn, recall_knn, _ = precision_recall_curve(y_test, y_prob_knn)
pr_auc_knn = auc(recall_knn, precision_knn)
results.append({
    "Model": "KNN",
    "Accuracy": accuracy_score(y_test, y_pred_knn),
    "Precision": precision_score(y_test, y_pred_knn, zero_division=0),
    "Recall": recall_score(y_test, y_pred_knn, zero_division=0),
    "F1": f1_score(y_test, y_pred_knn, zero_division=0),
    "ROC-AUC": roc_auc_score(y_test, y_pred_knn),
    "PR-AUC": pr_auc_knn
})

# 13) CatBoost
from catboost import CatBoostClassifier
cb = CatBoostClassifier(
    iterations=300,
    depth=6,
    learning_rate=0.1,
    loss_function="Logloss",
    random_seed=42,
    verbose=False,
    class_weights=[1, sum(y_train==0)/sum(y_train==1)]
)
cb.fit(X_train, y_train)
results.append(evaluate_model(cb, X_test, y_test, "CatBoost"))



In [ ]:
results_df = pd.DataFrame(results)

def try_add_probs(model, X, y, row_mask_name=None):
    try:
        if hasattr(model, "predict_proba"):
            prob = model.predict_proba(X)[:,1]
        elif hasattr(model, "decision_function"):
            prob = model.decision_function(X)
        else:
            return None, None
        roc = roc_auc_score(y, prob)
        precision, recall, _ = precision_recall_curve(y, prob)
        pr = auc(recall, precision)
        return roc, pr
    except Exception:
        return None, None

# Models mapping
trained_models = {
    "Logistic Regression": lr,
    "Decision Tree": dt,
    "Random Forest": rf,
    "XGBoost": xgb,
    "LightGBM": lgbm,
    "Naive Bayes": nb,
    "MLP Neural Network": mlp,
    "Extra Trees": et_model,
    "TabNet": tabnet_model,
    "KNN": knn_model,
    "CatBoost": cb
}

for i, row in results_df.iterrows():
    name = row['Model']
    if name in trained_models:
        roc, pr = try_add_probs(trained_models[name], X_test_scaled, y_test)
        if roc is not None and np.isnan(results_df.at[i, 'ROC-AUC']):
            results_df.at[i, 'ROC-AUC'] = roc
        if pr is not None and np.isnan(results_df.at[i, 'PR-AUC']):
            results_df.at[i, 'PR-AUC'] = pr

results_df.to_csv(os.path.join(BASE, "results_metrics.csv"), index=False)
results_df


In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

def get_preds_for_plot(model_name, model_obj):
    unscaled_models = ["Decision Tree", "Random Forest", "XGBoost", "LightGBM", "CatBoost"]
    if model_name in unscaled_models:
        X_for = X_test
    else:
        X_for = X_test_scaled
    if model_name == "Isolation Forest":
        return pred_iso
    if model_name == "One-Class SVM":
        return pred_ocsvm
    try:
        return model_obj.predict(X_for)
    except Exception:
        return model_obj.predict(X_test_scaled)

for name, model in trained_models.items():
    try:
        preds = get_preds_for_plot(name, model)
        cm = confusion_matrix(y_test, preds)
        plt.figure(figsize=(5,4))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
        plt.title(f"Confusion Matrix: {name}")
        plt.ylabel("True")
        plt.xlabel("Predicted")
        fname = f"confusion_matrices/{name.replace(' ', '_')}_confusion.png"
        plt.savefig(os.path.join(BASE, fname), bbox_inches='tight')
        plt.close()
    except Exception as e:
        print(f"Could not plot confusion for {name}: {e}")


cm_iso = confusion_matrix(y_test, pred_iso)
plt.figure(figsize=(5,4))
sns.heatmap(cm_iso, annot=True, fmt='d', cmap='Blues')
plt.title("Confusion Matrix: Isolation Forest")
plt.ylabel("True")
plt.xlabel("Predicted")
plt.savefig(os.path.join(BASE, "confusion_matrices", "Isolation_Forest_confusion.png"), bbox_inches='tight')
plt.close()

# One-Class SVM
cm_ocsvm = confusion_matrix(y_test, pred_ocsvm)
plt.figure(figsize=(5,4))
sns.heatmap(cm_ocsvm, annot=True, fmt='d', cmap='Blues')
plt.title("Confusion Matrix: One-Class SVM")
plt.ylabel("True")
plt.xlabel("Predicted")
plt.savefig(os.path.join(BASE, "confusion_matrices", "OneClassSVM_confusion.png"), bbox_inches='tight')
plt.close()


In [ ]:
plt.figure(figsize=(10,6))
plotted_any = False
for name, model in trained_models.items():
    try:
        unscaled = ["Decision Tree", "Random Forest", "XGBoost", "LightGBM", "CatBoost"]
        X_for = X_test if name in unscaled else X_test_scaled

        if hasattr(model, "predict_proba"):
            y_prob = model.predict_proba(X_for)[:,1]
        elif hasattr(model, "decision_function"):
            y_prob = model.decision_function(X_for)
        else:
            continue
        fpr, tpr, _ = roc_curve(y_test, y_prob)
        plt.plot(fpr, tpr, label=name)
        plotted_any = True
    except Exception as e:
        print(f"Skipping ROC for {name}: {e}")

if plotted_any:
    plt.plot([0,1],[0,1],'k--')
    plt.xlabel("FPR"); plt.ylabel("TPR"); plt.title("ROC Curves")
    plt.legend()
    plt.savefig(os.path.join(BASE, "plots", "roc_all_models.png"), bbox_inches='tight')
    plt.show()
else:
    print("No ROC curves plotted (no model provided probabilities).")

# Precision-Recall curves
plt.figure(figsize=(10,6))
plotted_any = False
for name, model in trained_models.items():
    try:
        unscaled = ["Decision Tree", "Random Forest", "XGBoost", "LightGBM", "CatBoost"]
        X_for = X_test if name in unscaled else X_test_scaled

        if hasattr(model, "predict_proba"):
            y_prob = model.predict_proba(X_for)[:,1]
        elif hasattr(model, "decision_function"):
            y_prob = model.decision_function(X_for)
        else:
            continue
        precision, recall, _ = precision_recall_curve(y_test, y_prob)
        plt.plot(recall, precision, label=name)
        plotted_any = True
    except Exception as e:
        print(f"Skipping PR for {name}: {e}")

if plotted_any:
    plt.xlabel("Recall"); plt.ylabel("Precision"); plt.title("Precision-Recall Curves")
    plt.legend()
    plt.savefig(os.path.join(BASE, "plots", "pr_all_models.png"), bbox_inches='tight')
    plt.show()
else:
    print("No PR curves plotted (no model provided probabilities).")


In [ ]:
numeric_cols = ["ROC-AUC","PR-AUC","F1"]
for c in numeric_cols:
    if c not in results_df.columns:
        results_df[c] = np.nan

plt.figure(figsize=(14,6))
plot_df = results_df.set_index("Model")[[ "ROC-AUC","PR-AUC","F1" ]].sort_values("ROC-AUC", ascending=False)
plot_df.plot(kind="bar", figsize=(14,6))
plt.title("Model Comparison on Fraud Detection (ROC-AUC, PR-AUC, F1)")
plt.ylabel("Score")
plt.xticks(rotation=45)
plt.grid(axis='y')
plt.savefig(os.path.join(BASE, "plots", "model_comparison_bar.png"), bbox_inches='tight')
plt.show()

plt.figure(figsize=(12,8))
heatmap_data = results_df.set_index("Model").astype(float)
sns.heatmap(heatmap_data, annot=True, fmt=".4f", cmap="YlGnBu")
plt.title("Model Comparison Heatmap (All Metrics)")
plt.ylabel("Model")
plt.xlabel("Metric")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(os.path.join(BASE, "plots", "heatmap_comparison.png"), bbox_inches='tight')
plt.show()

results_df.to_csv(os.path.join(BASE, "results_metrics_full.csv"), index=False)


In [ ]:
feature_names = X.columns.tolist()

def save_feature_importance(model, model_name, transform_needed=False):
    importance = None
    try:
        if hasattr(model, "feature_importances_"):
            importance = model.feature_importances_
        elif hasattr(model, "get_booster"):
            try:
                importance = model.get_booster().get_score(importance_type='gain')
                items = sorted(importance.items(), key=lambda x: x[1], reverse=True)[:30]
                names = [k for k,_ in items]
                vals = [v for _,v in items]
                plt.figure(figsize=(10,6))
                sns.barplot(x=vals, y=names)
                plt.title(f"Feature importance (top) - {model_name}")
                fname = f"feature_importance/{model_name.replace(' ', '_')}_importance.png"
                plt.tight_layout()
                plt.savefig(os.path.join(BASE, fname), bbox_inches='tight')
                plt.close()
                return
            except Exception:
                importance = None
        # CatBoost
        elif hasattr(model, "get_feature_importance"):
            vals = model.get_feature_importance()
            importance = np.array(vals)

        if importance is not None:
            if isinstance(importance, dict):
                items = sorted(importance.items(), key=lambda x: x[1], reverse=True)[:30]
                names = [k for k,_ in items]
                vals = [v for _,v in items]
                plt.figure(figsize=(10,6))
                sns.barplot(x=vals, y=names)
                plt.title(f"Feature importance (top) - {model_name}")
                fname = f"feature_importance/{model_name.replace(' ', '_')}_importance.png"
                plt.tight_layout()
                plt.savefig(os.path.join(BASE, fname), bbox_inches='tight')
                plt.close()
                return

            imp_arr = np.array(importance)
            if len(imp_arr) != len(feature_names):
                idxs = np.argsort(imp_arr)[-30:][::-1] if len(imp_arr) > 0 else []
                names = [f"f{int(i)}" for i in idxs]
                vals = imp_arr[idxs]
            else:
                idxs = np.argsort(imp_arr)[-30:][::-1]
                names = [feature_names[i] for i in idxs]
                vals = imp_arr[idxs]

            plt.figure(figsize=(10,6))
            sns.barplot(x=vals, y=names)
            plt.title(f"Feature importance (top) - {model_name}")
            fname = f"feature_importance/{model_name.replace(' ', '_')}_importance.png"
            plt.tight_layout()
            plt.savefig(os.path.join(BASE, fname), bbox_inches='tight')
            plt.close()
    except Exception as e:
        print(f"Could not compute feature importance for {model_name}: {e}")

# Random Forest
save_feature_importance(rf, "Random Forest")
# Extra Trees
save_feature_importance(et_model, "Extra Trees")
# XGBoost
save_feature_importance(xgb, "XGBoost")
# LightGBM
save_feature_importance(lgbm, "LightGBM")
# CatBoost
save_feature_importance(cb, "CatBoost")

print("Feature importance images saved to:", os.path.join(BASE, "feature_importance"))


In [ ]:
from google.colab import files

!zip -r /content/results.zip /content/results
!zip -r /content/catboost_info.zip /content/catboost_info

files.download("/content/results.zip")
files.download("/content/catboost_info.zip")


In [ ]:
pd1 = pd.read_csv("/content/results/dataset_stats.csv")
pd1

In [ ]:
pd2 = pd.read_csv("/content/results/results_metrics.csv")
pd2

In [ ]:
pd3 = pd.read_csv("/content/results/results_metrics_full.csv")
pd3

In [ ]:
!pip install -q xgboost lightgbm catboost imbalanced-learn pytorch-tabnet shap


In [ ]:
import os, gc, joblib, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix,
    precision_recall_curve, roc_curve, auc
)

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, IsolationForest
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier
from sklearn.svm import OneClassSVM

import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier
from pytorch_tabnet.tab_model import TabNetClassifier

from tqdm.auto import tqdm
import warnings
warnings.filterwarnings("ignore")

BASE = "/content/results"
os.makedirs(BASE, exist_ok=True)
os.makedirs(os.path.join(BASE, "plots"), exist_ok=True)
os.makedirs(os.path.join(BASE, "models"), exist_ok=True)

RANDOM_STATE = 42


In [ ]:
DATA_PATH = "/content/creditcard.csv"
df = pd.read_csv(DATA_PATH)
print("Loaded dataset shape:", df.shape)

if 'Class' in df.columns:
    df = df.rename(columns={'Class':'label'})

print("Columns:", df.columns.tolist())
display(df.head())

counts = df['label'].value_counts()
print("Class counts:\n", counts.to_dict())
print("Fraud rate: {:.6%}".format(counts.get(1,0)/len(df)))


In [ ]:
plt.figure(figsize=(6,4))
sns.countplot(x=df['label'])
plt.title("Class distribution")
plt.savefig(os.path.join(BASE, "plots", "class_distribution.png"), bbox_inches='tight')
plt.show()

# top correlations with label
corr = df.corr()['label'].abs().sort_values(ascending=False)
display(corr.head(12))
corr.to_csv(os.path.join(BASE, "plots", "corr_with_label.csv"))


In [ ]:
X = df.drop(columns=['label'])
y = df['label'].astype(int)

# Stratified split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)

# Scale numeric features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
joblib.dump(scaler, os.path.join(BASE, "models", "scaler.joblib"))

# SMOTE only for models that will use resampled balanced data
sm = SMOTE(random_state=RANDOM_STATE)
X_train_res, y_train_res = sm.fit_resample(X_train_scaled, y_train)

print("Shapes:")
print("X_train:", X_train.shape, "X_train_res:", X_train_res.shape, "X_test:", X_test.shape)


In [ ]:
def evaluate_model(model, X_for, y_true, model_name="Model"):
    preds = model.predict(X_for)
    # get prob/score
    probs = None
    if hasattr(model, "predict_proba"):
        probs = model.predict_proba(X_for)[:,1]
    elif hasattr(model, "decision_function"):
        probs = model.decision_function(X_for)

    acc = accuracy_score(y_true, preds)
    prec = precision_score(y_true, preds, zero_division=0)
    rec = recall_score(y_true, preds, zero_division=0)
    f1 = f1_score(y_true, preds, zero_division=0)
    roc = roc_auc_score(y_true, probs) if probs is not None else np.nan
    pr = average_precision_score(y_true, probs) if probs is not None else np.nan

    print(f"\n=== {model_name} ===\nAccuracy: {acc:.5f}, Precision: {prec:.5f}, Recall: {rec:.5f}, F1: {f1:.5f}, ROC-AUC: {roc:.5f}, PR-AUC: {pr:.5f}")
    print("Confusion Matrix:\n", confusion_matrix(y_true, preds))
    return {
        "Model": model_name, "Accuracy": acc, "Precision": prec, "Recall": rec,
        "F1": f1, "ROC-AUC": roc, "PR-AUC": pr
    }

results = []


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.model_selection import RandomizedSearchCV

print("Tuning Logistic Regression (fast)...")
pipe = Pipeline([("clf", LogisticRegression(solver="liblinear", max_iter=2000, random_state=RANDOM_STATE))])

param_dist = {
    "clf__penalty": ["l1", "l2"],
    "clf__C": [0.1, 1.0, 10.0],
    "clf__class_weight": [None, "balanced"]
}

lr_search = RandomizedSearchCV(
    pipe, param_dist, n_iter=6, scoring="average_precision", cv=StratifiedKFold(3, shuffle=True, random_state=RANDOM_STATE),
    n_jobs=-1, verbose=1, random_state=RANDOM_STATE
)
lr_search.fit(X_train_scaled, y_train)
best_lr = lr_search.best_estimator_
joblib.dump(best_lr, os.path.join(BASE, "models", "best_lr.joblib"))
results.append(evaluate_model(best_lr, X_test_scaled, y_test, "Logistic Regression"))


In [ ]:
from scipy.stats import randint as sp_randint

print("Tuning Random Forest (RandomizedSearch)...")
rf = RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1)

rf_param = {
    "n_estimators": [100, 200, 400],
    "max_depth": [None, 8, 16, 32],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "class_weight": [None, "balanced", "balanced_subsample"]
}

rf_search = RandomizedSearchCV(rf, rf_param, n_iter=25, scoring="average_precision", cv=3, random_state=RANDOM_STATE, n_jobs=-1, verbose=1)
rf_search.fit(X_train_scaled, y_train)
best_rf = rf_search.best_estimator_
joblib.dump(best_rf, os.path.join(BASE, "models", "best_rf.joblib"))
results.append(evaluate_model(best_rf, X_test_scaled, y_test, "Random Forest"))


In [ ]:
print("Tuning XGBoost (RandomizedSearch)...")
pos_weight = (y_train==0).sum() / max(1, (y_train==1).sum())

xgb_clf = xgb.XGBClassifier(objective="binary:logistic", use_label_encoder=False, eval_metric="logloss", random_state=RANDOM_STATE, n_jobs=-1)
xgb_param = {
    "n_estimators": [100, 200, 400],
    "max_depth": [3, 6, 10],
    "learning_rate": [0.01, 0.05, 0.1],
    "subsample": [0.6, 0.8, 1.0],
    "colsample_bytree": [0.6, 0.8, 1.0],
    "scale_pos_weight": [1, pos_weight]
}

xgb_search = RandomizedSearchCV(xgb_clf, xgb_param, n_iter=30, scoring="average_precision", cv=3, random_state=RANDOM_STATE, n_jobs=-1, verbose=1)
xgb_search.fit(X_train_scaled, y_train)
best_xgb = xgb_search.best_estimator_
joblib.dump(best_xgb, os.path.join(BASE, "models", "best_xgb.joblib"))
results.append(evaluate_model(best_xgb, X_test_scaled, y_test, "XGBoost"))


In [ ]:
print("Tuning LightGBM (RandomizedSearch)...")
lgb_clf = lgb.LGBMClassifier(random_state=RANDOM_STATE, n_jobs=-1)
lgb_param = {
    "n_estimators": [100, 200, 400],
    "num_leaves": [31, 64, 128],
    "max_depth": [-1, 8, 16],
    "learning_rate": [0.01, 0.05, 0.1],
    "subsample": [0.6, 0.8, 1.0],
    "class_weight": [None, "balanced"]
}

lgb_search = RandomizedSearchCV(lgb_clf, lgb_param, n_iter=25, scoring="average_precision", cv=3, random_state=RANDOM_STATE, n_jobs=-1, verbose=1)
lgb_search.fit(X_train_scaled, y_train)
best_lgb = lgb_search.best_estimator_
joblib.dump(best_lgb, os.path.join(BASE, "models", "best_lgb.joblib"))
results.append(evaluate_model(best_lgb, X_test_scaled, y_test, "LightGBM"))


In [ ]:
print("Tuning CatBoost (RandomizedSearch)...")
cb = CatBoostClassifier(task_type="CPU", verbose=False, random_seed=RANDOM_STATE)
cb_param = {
    "iterations": [100, 200, 400],
    "depth": [4,6,8],
    "learning_rate": [0.01, 0.05, 0.1],
    "l2_leaf_reg": [1, 3, 5]
}

cb_search = RandomizedSearchCV(cb, cb_param, n_iter=15, scoring="average_precision", cv=3, random_state=RANDOM_STATE, n_jobs=1, verbose=1)
cb_search.fit(X_train.values, y_train.values)
best_cb = cb_search.best_estimator_
joblib.dump(best_cb, os.path.join(BASE, "models", "best_cb.joblib"))
results.append(evaluate_model(best_cb, X_test.values, y_test, "CatBoost"))


In [ ]:
print("Tuning ExtraTrees (RandomizedSearch)...")
et = ExtraTreesClassifier(random_state=RANDOM_STATE, n_jobs=-1)
et_param = {
    "n_estimators": [100, 200, 400],
    "max_depth": [None, 8, 16],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 2],
    "class_weight": [None, "balanced"]
}

et_search = RandomizedSearchCV(et, et_param, n_iter=15, scoring="average_precision", cv=3, random_state=RANDOM_STATE, n_jobs=-1, verbose=1)
et_search.fit(X_train_res, y_train_res)
best_et = et_search.best_estimator_
joblib.dump(best_et, os.path.join(BASE, "models", "best_et.joblib"))
results.append(evaluate_model(best_et, X_test_scaled, y_test, "Extra Trees"))


In [ ]:
# KNN (GridSearch small grid)
print("Tuning KNN...")
from sklearn.model_selection import GridSearchCV
knn = KNeighborsClassifier()
knn_param = {"n_neighbors":[3,5,7], "weights":["uniform","distance"], "p":[1,2]}
gs_knn = GridSearchCV(knn, knn_param, scoring="average_precision", cv=3, n_jobs=-1, verbose=1)
gs_knn.fit(X_train_res, y_train_res)
best_knn = gs_knn.best_estimator_
joblib.dump(best_knn, os.path.join(BASE, "models", "best_knn.joblib"))
results.append(evaluate_model(best_knn, X_test_scaled, y_test, "KNN"))

# MLP RandomizedSearch
print("Tuning MLP...")
mlp = MLPClassifier(random_state=RANDOM_STATE, max_iter=1000)
mlp_param = {"hidden_layer_sizes":[(64,),(64,32),(128,64)], "alpha":[1e-4,1e-3,1e-2], "learning_rate_init":[1e-3,5e-4]}
mlp_search = RandomizedSearchCV(mlp, mlp_param, n_iter=12, scoring="average_precision", cv=3, random_state=RANDOM_STATE, n_jobs=-1, verbose=1)
mlp_search.fit(X_train_res, y_train_res)
best_mlp = mlp_search.best_estimator_
joblib.dump(best_mlp, os.path.join(BASE, "models", "best_mlp.joblib"))
results.append(evaluate_model(best_mlp, X_test_scaled, y_test, "MLP"))


In [ ]:
print("Training Naive Bayes (baseline)...")
nb = GaussianNB()
nb.fit(X_train_res, y_train_res)
joblib.dump(nb, os.path.join(BASE, "models", "nb.joblib"))
results.append(evaluate_model(nb, X_test_scaled, y_test, "Naive Bayes"))


In [ ]:
# Isolation Forest
print("Training Isolation Forest...")
contamination = y_train.mean()
iso = IsolationForest(contamination=contamination, random_state=RANDOM_STATE)
iso.fit(X_train_scaled[y_train==0])
pred_iso = iso.predict(X_test_scaled)
pred_iso = np.array([1 if p==-1 else 0 for p in pred_iso])
results.append({
    "Model":"Isolation Forest",
    "Accuracy": accuracy_score(y_test, pred_iso),
    "Precision": precision_score(y_test, pred_iso, zero_division=0),
    "Recall": recall_score(y_test, pred_iso, zero_division=0),
    "F1": f1_score(y_test, pred_iso, zero_division=0),
    "ROC-AUC": np.nan, "PR-AUC": np.nan
})

# One-Class SVM
print("Training One-Class SVM...")
ocsvm = OneClassSVM(kernel="rbf", nu=0.01, gamma="scale")
ocsvm.fit(X_train_scaled[y_train==0])
pred_ocsvm = ocsvm.predict(X_test_scaled)
pred_ocsvm = np.array([1 if p==-1 else 0 for p in pred_ocsvm])
results.append({
    "Model":"One-Class SVM",
    "Accuracy": accuracy_score(y_test, pred_ocsvm),
    "Precision": precision_score(y_test, pred_ocsvm, zero_division=0),
    "Recall": recall_score(y_test, pred_ocsvm, zero_division=0),
    "F1": f1_score(y_test, pred_ocsvm, zero_division=0),
    "ROC-AUC": np.nan, "PR-AUC": np.nan
})


In [ ]:
print("Training TabNet (on SMOTE-resampled scaled data)...")
tabnet = TabNetClassifier(seed=RANDOM_STATE, verbose=10)
tabnet.fit(
    X_train_res, y_train_res,
    eval_set=[(X_test_scaled, y_test)],
    max_epochs=50,
    patience=10,
    batch_size=1024,
    virtual_batch_size=128
)
joblib.dump(tabnet, os.path.join(BASE, "models", "tabnet.joblib"))
y_pred_tab = tabnet.predict(X_test_scaled)
results.append({
    "Model":"TabNet",
    "Accuracy": accuracy_score(y_test, y_pred_tab),
    "Precision": precision_score(y_test, y_pred_tab, zero_division=0),
    "Recall": recall_score(y_test, y_pred_tab, zero_division=0),
    "F1": f1_score(y_test, y_pred_tab, zero_division=0),
    "ROC-AUC": np.nan, "PR-AUC": np.nan
})


In [ ]:
results_df = pd.DataFrame(results)

trained_models = {
    "Logistic Regression": best_lr,
    "Random Forest": best_rf,
    "XGBoost": best_xgb,
    "LightGBM": best_lgb,
    "CatBoost": best_cb,
    "Extra Trees": best_et,
    "KNN": best_knn,
    "MLP": best_mlp,
    "Naive Bayes": nb,
    "TabNet": tabnet
}

def try_probs(model, X_for):
    try:
        if hasattr(model, "predict_proba"):
            return model.predict_proba(X_for)[:,1]
        elif hasattr(model, "decision_function"):
            return model.decision_function(X_for)
    except Exception:
        return None

for i,row in results_df.iterrows():
    name = row['Model']
    if name in trained_models:
        model = trained_models[name]
        X_for = X_test_scaled if name != "CatBoost" else X_test.values
        probs = try_probs(model, X_for)
        if probs is not None:
            try:
                results_df.at[i,'ROC-AUC'] = roc_auc_score(y_test, probs)
                p, r, _ = precision_recall_curve(y_test, probs)
                results_df.at[i,'PR-AUC'] = auc(r, p)
            except Exception:
                pass

results_df.to_csv(os.path.join(BASE, "results_metrics.csv"), index=False)
display(results_df.sort_values(by="PR-AUC", ascending=False))


In [ ]:
plt.figure(figsize=(10,8))
for name, model in trained_models.items():
    X_for = X_test_scaled if name != "CatBoost" else X_test.values
    probs = try_probs(model, X_for)
    if probs is None:
        continue
    fpr,tpr,_ = roc_curve(y_test, probs)
    plt.plot(fpr, tpr, label=f"{name} (AUC={roc_auc_score(y_test, probs):.3f})")
plt.plot([0,1],[0,1],'k--')
plt.xlabel("FPR"); plt.ylabel("TPR"); plt.title("ROC Curves")
plt.legend()
plt.savefig(os.path.join(BASE, "plots", "roc_all.png"), bbox_inches='tight')
plt.show()

plt.figure(figsize=(10,8))
for name, model in trained_models.items():
    X_for = X_test_scaled if name != "CatBoost" else X_test.values
    probs = try_probs(model, X_for)
    if probs is None:
        continue
    p,r,_ = precision_recall_curve(y_test, probs)
    plt.plot(r, p, label=f"{name} (PR-AUC={auc(r,p):.3f})")
plt.xlabel("Recall"); plt.ylabel("Precision"); plt.title("Precision-Recall Curves")
plt.legend()
plt.savefig(os.path.join(BASE, "plots", "pr_all.png"), bbox_inches='tight')
plt.show()


In [ ]:
top_names = list(trained_models.keys())
for name in top_names:
    model = trained_models[name]
    X_for = X_test_scaled if name != "CatBoost" else X_test.values
    try:
        preds = model.predict(X_for)
    except Exception as e:
        print(f"Skipping {name}: {e}")
        continue
    cm = confusion_matrix(y_test, preds)
    plt.figure(figsize=(4,3))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title(f"Confusion: {name}")
    plt.ylabel("True"); plt.xlabel("Pred")
    plt.tight_layout()
    plt.savefig(os.path.join(BASE, f"confusion_{name.replace(' ','_')}.png"), bbox_inches='tight')
    plt.show()


In [ ]:
def plot_fi(m, cols, name, topk=30):
    if hasattr(m, "feature_importances_"):
        fi = m.feature_importances_
        idx = np.argsort(fi)[-topk:]
        plt.figure(figsize=(8,6))
        plt.barh(np.array(cols)[idx], fi[idx])
        plt.title(f"Feature importance: {name}")
        plt.tight_layout()
        plt.savefig(os.path.join(BASE, f"fi_{name.replace(' ','_')}.png"), bbox_inches='tight')
        plt.show()

cols = X.columns.tolist()
for name in ["Random Forest","XGBoost","LightGBM","CatBoost","Extra Trees"]:
    if name in trained_models:
        plot_fi(trained_models[name], cols, name)


In [ ]:
joblib.dump(trained_models, os.path.join(BASE, "models", "best_models_collection.joblib"))
results_df.to_csv(os.path.join(BASE, "results_metrics_final.csv"), index=False)

print("Saved models and metrics to:", BASE)
display(results_df.sort_values(by="PR-AUC", ascending=False))


**XGBOOST**

In [ ]:
!pip install xgboost==1.7.6

In [ ]:
!pip install imbalanced-learn


In [ ]:
!pip install -q scikit-learn==1.3.2
!pip install -q xgboost==1.7.6
import sklearn
print("SKLEARN VERSION:", sklearn.__version__)


In [ ]:
from tqdm.auto import tqdm
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
import xgboost as xgb

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, average_precision_score,
    confusion_matrix, ConfusionMatrixDisplay
)
from imblearn.over_sampling import SMOTE

import warnings
warnings.filterwarnings("ignore")

RANDOM_STATE = 42

In [ ]:
print("📂 Loading dataset...")

DATA_PATH = "/content/creditcard.csv"
df = pd.read_csv(DATA_PATH)

df = df.rename(columns={"Class": "label"})
X = df.drop(columns=["label"])
y = df["label"].astype(int)

print("Dataset shape:", df.shape)
print("Fraud count:", y.sum())


In [ ]:
# SPLIT + SCALE + SMOTE
Print("\n🔄 Splitting & Scaling...")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

sm = SMOTE(random_state=RANDOM_STATE)
X_train_res, y_train_res = sm.fit_resample(X_train_scaled, y_train)

In [ ]:
step_timer = {}

def start_step(name):
    step_timer[name] = time.time()
    print(f"\n⏳ Starting: {name}")

def end_step(name):
    elapsed = time.time() - step_timer[name]
    print(f"✅ Finished: {name} — ({elapsed:.2f} sec)\n")

In [ ]:
# XGBOOST TUNING
start_step("XGBoost Hyperparameter Tuning")

pos_weight = (y_train == 0).sum() / max(1, (y_train == 1).sum())

xgb_clf = xgb.XGBClassifier(
    objective="binary:logistic",
    tree_method="gpu_hist",
    predictor="gpu_predictor",
    eval_metric="logloss",
    random_state=RANDOM_STATE,
    n_jobs=-1
)

param_grid = {
    "n_estimators": [200, 400, 600, 800],
    "max_depth": [4, 6, 8, 10],
    "learning_rate": [0.005, 0.01, 0.03, 0.1],
    "subsample": [0.6, 0.8, 1.0],
    "colsample_bytree": [0.6, 0.8, 1.0],
    "gamma": [0, 1, 5],
    "min_child_weight": [1, 3, 5],
    "scale_pos_weight": [1, pos_weight]
}

with tqdm(total=1, desc="RandomizedSearchCV Running", colour="green") as pbar:
    xgb_search = RandomizedSearchCV(
        xgb_clf,
        param_grid,
        n_iter=30,
        scoring="average_precision",
        cv=3,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbose=2
    )
    xgb_search.fit(X_train_scaled, y_train)
    pbar.update(1)

best_xgb = xgb_search.best_estimator_

print("\n🔥 Best Parameters:")
print(xgb_search.best_params_)

end_step("XGBoost Hyperparameter Tuning")

In [ ]:
# EVALUATION
start_step("Evaluation")

y_pred = best_xgb.predict(X_test_scaled)
y_prob = best_xgb.predict_proba(X_test_scaled)[:, 1]

acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc = roc_auc_score(y_test, y_prob)
pr_auc = average_precision_score(y_test, y_prob)

print("\n================= BEST XGBOOST METRICS =================")
print(f"Accuracy       : {acc:.5f}")
print(f"Precision      : {prec:.5f}")
print(f"Recall         : {rec:.5f}")
print(f"F1 Score       : {f1:.5f}")
print(f"ROC-AUC        : {roc:.5f}")
print(f"PR-AUC         : {pr_auc:.5f}")
print("========================================================")

end_step("Evaluation")

In [ ]:
# CONFUSION MATRIX
start_step("Confusion Matrix Plot")

cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
plt.figure(figsize=(6, 5))
disp.plot(cmap="Blues", values_format="d")
plt.title("Confusion Matrix: XGBoost")
plt.grid(False)
plt.show()

end_step("Confusion Matrix Plot")




In [ ]:
# FEATURE IMPORTANCE PLOT
start_step("Feature Importance Plot")

importances = best_xgb.feature_importances_
feature_names = X_train.columns

sorted_idx = np.argsort(importances)[::-1]
top_k = 20

plt.figure(figsize=(10, 8))
plt.barh(
    feature_names[sorted_idx][:top_k][::-1],
    importances[sorted_idx][:top_k][::-1]
)
plt.xlabel("Importance")
plt.title("XGBoost Feature Importance (Top 20)")
plt.tight_layout()
plt.show()

end_step("Feature Importance Plot")

**23-11-25**

In [ ]:
!pip install -q xgboost==1.7.6 lightgbm catboost imbalanced-learn pytorch-tabnet joblib tqdm


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.3/200.3 MB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.5/44.5 kB 2.9 MB/s eta 0:00:00


In [ ]:
import os, time, zipfile, joblib
from pathlib import Path
from tqdm.auto import tqdm
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")

import sklearn
from sklearn.model_selection import train_test_split, RandomizedSearchCV, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, average_precision_score, confusion_matrix,
                             roc_curve, precision_recall_curve, auc)
from imblearn.over_sampling import SMOTE

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import IsolationForest
from sklearn.svm import OneClassSVM

import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier
from pytorch_tabnet.tab_model import TabNetClassifier

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 120)

BASE = "/content/results"
plots_dir = os.path.join(BASE, "plots")
models_dir = os.path.join(BASE, "models")
conf_dir = os.path.join(BASE, "confusion_matrices")
fi_dir = os.path.join(BASE, "feature_importance")
per_model_base = os.path.join(BASE, "by_model")

for d in [BASE, plots_dir, models_dir, conf_dir, fi_dir, per_model_base]:
    Path(d).mkdir(parents=True, exist_ok=True)

print("Setup done. Results will be saved in:", BASE)
print("SKLEARN VERSION:", sklearn.__version__)


Setup done. Results will be saved in: /content/results
SKLEARN VERSION: 1.6.1


In [ ]:
DATA_PATH = "/content/creditcard.csv"
RANDOM_STATE = 42

print("Loading dataset:", DATA_PATH)
df = pd.read_csv(DATA_PATH)
print("Initial shape:", df.shape)

# unify label name if needed
if 'Class' in df.columns:
    df = df.rename(columns={'Class': 'label'})
elif 'Label' in df.columns:
    df = df.rename(columns={'Label': 'label'})

if 'label' not in df.columns:
    raise ValueError("ERROR: Label column not found. Expected column 'Class' or 'Label'.")

invalid_rows = df['label'].isna().sum() + np.isinf(df['label']).sum()
print(f"Rows with invalid labels: {invalid_rows}")

df = df[pd.to_numeric(df['label'], errors='coerce').notnull()]
df['label'] = df['label'].astype(int)

print("After cleaning label column, shape:", df.shape)

print("Columns:", df.columns.tolist())
print("Class distribution:\n", df['label'].value_counts())
print("Fraud rate: {:.6%}".format(df['label'].mean()))

# Features/target
X = df.drop(columns=['label'])
y = df['label']

# Train-test split (raw unscaled kept for tree models & CatBoost)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

# Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Save scaler
joblib.dump(scaler, os.path.join(models_dir, "scaler.joblib"))

# SMOTE for models that require balanced + scaled inputs
sm = SMOTE(random_state=RANDOM_STATE, k_neighbors=5)
X_train_res, y_train_res = sm.fit_resample(X_train_scaled, y_train)

print("\nShapes:")
print("X_train (raw):", X_train.shape)
print("X_train_scaled:", X_train_scaled.shape)
print("X_train_res (scaled+SMOTE):", X_train_res.shape)
print("X_test:", X_test.shape)


Loading dataset: /content/creditcard.csv
Initial shape: (11959, 31)
Rows with invalid labels: 1
After cleaning label column, shape: (11958, 31)
Columns: ['Time', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Amount', 'label']
Class distribution:
 label
0    11906
1       52
Name: count, dtype: int64
Fraud rate: 0.434855%

Shapes:
X_train (raw): (9566, 30)
X_train_scaled: (9566, 30)
X_train_res (scaled+SMOTE): (19048, 30)
X_test: (2392, 30)


In [ ]:

def ensure_model_folder(model_name):
    p = os.path.join(per_model_base, model_name.replace(" ", "_"))
    Path(p).mkdir(parents=True, exist_ok=True)
    return p

def evaluate_and_save(model, X_for_eval, model_name, used_scaling=True, used_smote=False):
    """Evaluate, print, save metrics & confusion matrix and returns metrics dict."""
    folder = ensure_model_folder(model_name)
    print(f"\n--- Evaluating {model_name} ---")
    preds = model.predict(X_for_eval)
    probs = None
    if hasattr(model, "predict_proba"):
        try:
            probs = model.predict_proba(X_for_eval)[:,1]
        except Exception:
            probs = None
    elif hasattr(model, "decision_function"):
        try:
            probs = model.decision_function(X_for_eval)
        except Exception:
            probs = None

    acc = accuracy_score(y_test, preds)
    prec = precision_score(y_test, preds, zero_division=0)
    rec = recall_score(y_test, preds, zero_division=0)
    f1 = f1_score(y_test, preds, zero_division=0)
    roc = roc_auc_score(y_test, probs) if probs is not None else np.nan
    pr = average_precision_score(y_test, probs) if probs is not None else np.nan


    print(f"Accuracy: {acc:.5f}, Precision: {prec:.5f}, Recall: {rec:.5f}, F1: {f1:.5f}")
    print(f"ROC-AUC: {roc if not np.isnan(roc) else 'NA'}, PR-AUC: {pr if not np.isnan(pr) else 'NA'}")
    cm = confusion_matrix(y_test, preds)
    print("Confusion Matrix:\n", cm)

    metrics = {
        "Model": model_name, "Accuracy": acc, "Precision": prec, "Recall": rec,
        "F1": f1, "ROC-AUC": roc if not np.isnan(roc) else None, "PR-AUC": pr if not np.isnan(pr) else None,
        "Used_Scaling": used_scaling, "Used_SMOTE": used_smote
    }
    pd.DataFrame([metrics]).to_csv(os.path.join(folder, "metrics.csv"), index=False)

    plt.figure(figsize=(4,3))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["Normal","Fraud"], yticklabels=["Normal","Fraud"])
    plt.title(f"Confusion Matrix: {model_name}")
    plt.ylabel("True")
    plt.xlabel("Predicted")
    cm_path = os.path.join(folder, f"{model_name.replace(' ','_')}_confusion.png")
    plt.tight_layout()
    plt.savefig(cm_path, bbox_inches='tight')
    plt.close()

    try:
        joblib.dump(model, os.path.join(folder, f"{model_name.replace(' ','_')}.joblib"))
    except Exception as e:
        print(f"Could not joblib.dump for {model_name}: {e}")

    return metrics

def plot_and_save_feature_importance(model, model_name, feature_names, topk=30):
    folder = ensure_model_folder(model_name)
    fname = os.path.join(folder, f"{model_name.replace(' ','_')}_feature_importance.png")

    imp = None
    try:
        if hasattr(model, "feature_importances_"):
            imp = np.array(model.feature_importances_)
        elif hasattr(model, "get_booster"):

            try:
                booster = model.get_booster()
                score = booster.get_score(importance_type='gain')
                items = sorted(score.items(), key=lambda x: x[1], reverse=True)
                names = [k for k,_ in items]
                vals = [v for _,v in items]
                plt.figure(figsize=(8, max(4, len(names)*0.2)))
                sns.barplot(x=vals[:topk], y=names[:topk])
                plt.title(f"Feature importance (gain) - {model_name}")
                plt.tight_layout()
                plt.savefig(fname, bbox_inches='tight')
                plt.close()
                return
            except Exception:
                imp = None
        elif hasattr(model, "get_feature_importance"):
            try:
                imp = np.array(model.get_feature_importance())
            except Exception:
                imp = None
    except Exception as e:
        print("FI extract exception:", e)
        imp = None

    if imp is None:
        print(f"No feature importance available for {model_name}")
        return

    imp_arr = np.asarray(imp)
    if imp_arr.shape[0] != len(feature_names):
        idxs = np.argsort(imp_arr)[-topk:][::-1]
        names = [f"f{int(i)}" for i in idxs]
        vals = imp_arr[idxs]
    else:
        idxs = np.argsort(imp_arr)[-topk:][::-1]
        names = [feature_names[i] for i in idxs]
        vals = imp_arr[idxs]

    plt.figure(figsize=(8, max(4, len(names)*0.2)))
    sns.barplot(x=vals, y=names)
    plt.title(f"Feature importance: {model_name}")
    plt.tight_layout()
    plt.savefig(fname, bbox_inches='tight')
    plt.close()

def try_get_probs(model, X_for_eval):
    try:
        if hasattr(model, "predict_proba"):
            return model.predict_proba(X_for_eval)[:,1]
        elif hasattr(model, "decision_function"):
            return model.decision_function(X_for_eval)
    except Exception:
        return None


In [ ]:
# Logistic Regression tuning (scaled + SMOTE)
print("Logistic Regression tuning (scaled + SMOTE)")

model_name = "Logistic Regression"
folder = ensure_model_folder(model_name)

param_dist = {
    "penalty": ["l1", "l2", "elasticnet"],
    "C": [0.01, 0.05, 0.1, 0.5, 1.0, 5.0],
    "solver": ["saga"],
    "l1_ratio": [None, 0.15, 0.5]
}

lr = LogisticRegression(max_iter=5000, random_state=RANDOM_STATE, class_weight="balanced")
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

rs = RandomizedSearchCV(lr, param_distributions=param_dist, n_iter=30, scoring="average_precision",
                        cv=cv, random_state=RANDOM_STATE, n_jobs=-1, verbose=1)

start = time.time()
rs.fit(X_train_res, y_train_res)
end = time.time()
print(f"Tuned in {(end-start):.1f}s. Best params:", rs.best_params_)

best_lr = rs.best_estimator_
metrics_lr = evaluate_and_save(best_lr, X_test_scaled, model_name, used_scaling=True, used_smote=True)
plot_and_save_feature_importance(best_lr, model_name, feature_names=X.columns.tolist())
